# Khảo Sát Tập Hành Động UCF-101 (Phase 0.4 Action UCF-101 Explore)

Tài liệu này thực hiện các phân tích thăm dò định lượng đối với tập dữ liệu UCF-101, tập trung vào cấu trúc Nhóm phân chia chuẩn (Official Splits) và các chỉ số hình học khung hình. 
Áp dụng nguyên lý kiến trúc Fail-safe: Quá trình thẩm định (validation) vẫn được triển khai dưới dạng Khung Logic giả lập mà không gây phát sinh biệt lệ (Exceptions) trong trường hợp dữ liệu thô chưa được ánh xạ trong kho lưu trữ `data/external`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.training.scripts.data_utils import (
    build_action_manifest,
    load_yaml,
    sample_frame_indices,
)

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (10, 6), "axes.titlesize": 14})

repo = Path("..").resolve()
config = load_yaml(repo / "configs/datasets/action_accessibility.yaml")
videos_root = repo / config["base_dataset"]["videos_root"]
split_root = repo / config["base_dataset"]["split_root"]
classes = config["classes"]["public_bootstrap"]

print(f"Target path: {videos_root}")
print(f"Split root: {split_root}")

## 1. Thành Lập Bộ Chỉ Mục (Manifest Construction)
Hàm thức cấu trúc sẽ duyệt qua cây thư mục vật lý để ánh xạ số liệu vào Bảng Khai Báo (Manifest). Nó cũng tích hợp việc tính toán sự xuất bản của các thành phần đối tượng (video clips) thuộc quỹ đạo nhóm hành động tiêu biểu.

In [ ]:
if videos_root.exists():
    rows = build_action_manifest(
        videos_root,
        classes,
        dataset_name=config["base_dataset"]["name"],
        split_root=split_root,
    )

    df_manifest = pd.DataFrame(rows)
    print(f"Tạo thành công Manifest với {len(rows)} bản ghi.")
    if not df_manifest.empty:
        display(df_manifest.head())
else:
    df_manifest = pd.DataFrame()
    print("Hệ thống không tìm thấy nguyên liệu UCF-101 tại data/external/ucf101/.")

## 2. Trực Quan Hóa Mật Độ Phân Lớp (Class Density Visualization)
Cơ chế đồ họa dưới đây phân tích dữ liệu thống kê từ cột `label` của bảng Manifest nhằm biểu hiện tương quan quy mô (volume correlation) của các hạng mục. Việc quan sát Mật độ Lớp giúp ước lượng sai số mô hình (Model Bias) có thể phát sinh.

In [ ]:
if not df_manifest.empty:
    class_counts = df_manifest["label"].value_counts().reset_index()
    class_counts.columns = ["Phân Lớp (Class)", "Tần Suất (Count)"]

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=class_counts,
        x="Phân Lớp (Class)",
        y="Tần Suất (Count)",
        color="darkorange",
    )
    plt.xticks(rotation=45, ha="right")
    plt.title(
        "Tương Quan Tần Suất Các Hạng Mục Hành Động (Action Categories Correlation)"
    )
    plt.ylabel("Số Lượng Khung Phim (Video Count)")
    plt.tight_layout()
    plt.show()
else:
    print("Bỏ qua Barplot vì kích thước dữ liệu = 0.")

## 3. Khảo Sát Phân Bố Mẫu Khung Hình (Frame Sample Distribution)
Khảo sát quỹ đạo hàm định danh `sample_frame_indices`. Một ma trận mảng chứa các khoảng cách phân bổ (step distributions) được thể hiện tĩnh bằng Biểu đồ Tần suất (Histogram).

In [ ]:
frames_to_sample = 40
sample_indices = sample_frame_indices(frames_to_sample)

plt.figure(figsize=(8, 4))
sns.histplot(sample_indices, bins=20, kde=True, color="teal")
plt.title(f"Mật Độ Tần Suất Xếp Mẫu Khung Hình (N={frames_to_sample} Khung)")
plt.xlabel("Giao Chỉ Số Khung Hình (Frame Index)")
plt.ylabel("Biên Độ Mật Độ (Density Magnitude)")
plt.show()

display(pd.DataFrame({"Mẫu Trích Xuất Cơ Sở": [sample_indices]}))